# Lesson 3: Building an Agent Reasoning Loop

## Setup

In [1]:
import nest_asyncio
nest_asyncio.apply()

## Load the data

To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

**Note**: The pdf file is included with this lesson. To access it, go to the `File` menu and select`Open...`.

## Setup the Query Tools

In [2]:
! pip install -q llama-index llama-index-llms-anthropic llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.4/198.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.0/189.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.6/82.6 kB 5.3 MB/s eta 0:00:00


In [3]:
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.agent import AgentRunner

In [4]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, SummaryIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool, FunctionTool
from llama_index.core.vector_stores import MetadataFilters, FilterCondition
from typing import List, Optional

In [5]:
import os

os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"
os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"

In [ ]:
llm = Anthropic(model="claude-3-haiku-20240307")
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

In [12]:
Settings.llm = llm
Settings.embed_model = embed_model

In [13]:
def get_doc_tools(
    file_path: str,
    name: str,
) -> str:
    """Get vector query and summary query tools from a document."""

    # load documents
    documents = SimpleDirectoryReader(input_files=[file_path]).load_data()
    splitter = SentenceSplitter(chunk_size=1024)
    nodes = splitter.get_nodes_from_documents(documents)
    vector_index = VectorStoreIndex(nodes)

    def vector_query(
        query: str,
        page_numbers: Optional[List[str]] = None
    ) -> str:
        """Use to answer questions over the MetaGPT paper.

        Useful if you have specific questions over the MetaGPT paper.
        Always leave page_numbers as None UNLESS there is a specific page you want to search for.

        Args:
            query (str): the string query to be embedded.
            page_numbers (Optional[List[str]]): Filter by set of pages. Leave as NONE
                if we want to perform a vector search
                over all pages. Otherwise, filter by the set of specified pages.

        """

        page_numbers = page_numbers or []
        metadata_dicts = [
            {"key": "page_label", "value": p} for p in page_numbers
        ]

          query_engine = vector_index.as_query_engine(
            similarity_top_k=2,
            filters=MetadataFilters.from_dicts(
                metadata_dicts,
                condition=FilterCondition.OR
            )
        )
        response = query_engine.query(query)
        return response


    vector_query_tool = FunctionTool.from_defaults(
        name=f"vector_tool_{name}",
        fn=vector_query
    )

    summary_index = SummaryIndex(nodes)
    summary_query_engine = summary_index.as_query_engine(
        response_mode="tree_summarize",
        use_async=True,
    )
    summary_tool = QueryEngineTool.from_defaults(
        name=f"summary_tool_{name}",
        query_engine=summary_query_engine,
        description=(
            "Use ONLY IF you want to get a holistic summary of MetaGPT. "
            "Do NOT use if you have specific questions over MetaGPT."
        ),
    )

    return vector_query_tool, summary_tool

In [14]:
vector_tool, summary_tool = get_doc_tools("metagpt.pdf", "metagpt")

## Configurar agente de chamada de função

In [15]:
agent_worker = FunctionCallingAgentWorker.from_tools(
    [vector_tool, summary_tool],
    llm=llm,
    verbose=True
)
agent = AgentRunner(agent_worker)

In [16]:
response = agent.query(
    "Fale-me sobre as funções dos agentes no MetaGPT, "
    "e como eles se comunicam entre si."
)

Added user message to memory: Fale-me sobre as funções dos agentes no MetaGPT, e como eles se comunicam entre si.
=== Calling Function ===
Calling function: vector_tool_metagpt with args: {"query": "fun\u00e7\u00f5es dos agentes no MetaGPT e como eles se comunicam"}
=== Function Output ===
O MetaGPT utiliza uma abordagem estruturada de comunicação entre agentes, onde cada agente tem um papel e responsabilidades específicas. Por exemplo:

- O Arquiteto gera dois outputs principais: o design da interface do sistema e um diagrama de fluxo de sequência. Esses artefatos contêm o design dos módulos do sistema e as sequências de interação, que servem como entregas importantes para os Engenheiros.

- Os agentes se comunicam através de documentos e diagramas (outputs estruturados) em vez de diálogo. Esses documentos contêm todas as informações necessárias, evitando conteúdo irrelevante ou ausente.

Além disso, o MetaGPT utiliza um mecanismo de publicação-assinatura para compartilhar informações

In [17]:
print(response.source_nodes[0].get_content(metadata_mode="all"))

page_label: 6
file_name: metagpt.pdf
file_path: metagpt.pdf
file_type: application/pdf
file_size: 16911937
creation_date: 2024-11-16
last_modified_date: 2024-11-16

Preprint
whispers)2, after several rounds of communication, the original information may be quite distorted.
Inspired by human social structures, we propose using structured communication to formulate the
communication of agents. We establish a schema and format for each role and request that individ-
uals provide the necessary outputs based on their specific role and context.
As shown in Figure 3, the Architect agent generates two outputs: the system interface design and a
sequence flow diagram. These contain system module design and interaction sequences, which serve
as important deliverables for Engineers. Unlike ChatDev (Zhao et al., 2023), agents in MetaGPT
communicate through documents and diagrams (structured outputs) rather than dialogue. These
documents contain all necessary information, preventing irrelevant or mi

In [18]:
response = agent.chat(
    "Fale-me sobre os conjuntos de dados de avaliação usados."
)

Added user message to memory: Fale-me sobre os conjuntos de dados de avaliação usados.
=== Calling Function ===
Calling function: vector_tool_metagpt with args: {"query": "conjuntos de dados de avalia\u00e7\u00e3o usados no MetaGPT"}
=== Function Output ===
O contexto fornecido não menciona conjuntos de dados de avaliação usados no MetaGPT. O texto se concentra em detalhes sobre o dataset de desenvolvimento de software utilizado nos experimentos do MetaGPT, incluindo uma tabela com 11 tarefas desse dataset. Não há informações sobre outros conjuntos de dados de avaliação usados no sistema.
=== LLM Response ===
Infelizmente, o texto fornecido não menciona detalhes sobre os conjuntos de dados de avaliação utilizados no MetaGPT. O foco está na descrição do dataset de desenvolvimento de software usado nos experimentos, mas não há informações sobre outros conjuntos de dados de avaliação empregados. Portanto, não posso fornecer mais detalhes sobre esse aspecto com base no material disponível.

In [19]:
response = agent.chat("Tell me the results over one of the above datasets.")

Added user message to memory: Tell me the results over one of the above datasets.
=== LLM Response ===
Unfortunately, the information provided does not contain details about the evaluation results over the datasets used in the MetaGPT system. The text focuses on describing the software development dataset used in the experiments, but does not provide any quantitative results or performance metrics over that or any other evaluation datasets. Without access to those specific details from the paper, I am unable to summarize the results obtained on the evaluation datasets. The information given is limited to the high-level description of the dataset and the overall approach, but does not include the actual evaluation outcomes. I apologize that I cannot provide more specifics on the results over the datasets based on the material available to me.


## Nível inferior: Depurabilidade e controle

In [20]:
agent_worker = FunctionCallingAgentWorker.from_tools(
    [vector_tool, summary_tool],
    llm=llm,
    verbose=True
)
agent = AgentRunner(agent_worker)

In [21]:
task = agent.create_task(
    "Fale-me sobre as funções dos agentes no MetaGPT, "
    "e como eles se comunicam entre si."
)

In [22]:
step_output = agent.run_step(task.task_id)

Added user message to memory: Fale-me sobre as funções dos agentes no MetaGPT, e como eles se comunicam entre si.
=== Calling Function ===
Calling function: vector_tool_metagpt with args: {"query": "fun\u00e7\u00f5es dos agentes no MetaGPT e como eles se comunicam"}
=== Function Output ===
O MetaGPT utiliza uma abordagem estruturada de comunicação entre agentes, onde cada agente tem um papel e responsabilidades específicas. Por exemplo:

- O Arquiteto gera dois outputs principais: o design da interface do sistema e um diagrama de fluxo de sequência. Esses artefatos contêm o design dos módulos do sistema e as sequências de interação, que servem como entregas importantes para os Engenheiros.

- Os agentes se comunicam através de documentos e diagramas (outputs estruturados) em vez de diálogo. Esses documentos contêm todas as informações necessárias, evitando conteúdo irrelevante ou faltante.

Além disso, o MetaGPT utiliza um mecanismo de publicação-assinatura para compartilhar informaçõe

In [23]:
completed_steps = agent.get_completed_steps(task.task_id)
print(f"Num completed for task {task.task_id}: {len(completed_steps)}")
print(completed_steps[0].output.sources[0].raw_output)

Num completed for task 21fe1ba2-82bc-4e4e-af8f-a0aad7d53606: 1
O MetaGPT utiliza uma abordagem estruturada de comunicação entre agentes, onde cada agente tem um papel e responsabilidades específicas. Por exemplo:

- O Arquiteto gera dois outputs principais: o design da interface do sistema e um diagrama de fluxo de sequência. Esses artefatos contêm o design dos módulos do sistema e as sequências de interação, que servem como entregas importantes para os Engenheiros.

- Os agentes se comunicam através de documentos e diagramas (outputs estruturados) em vez de diálogo. Esses documentos contêm todas as informações necessárias, evitando conteúdo irrelevante ou faltante.

Além disso, o MetaGPT utiliza um mecanismo de publicação-assinatura para compartilhar informações de maneira eficiente. Os agentes publicam suas mensagens estruturadas em um pool de mensagens global, e podem acessar as mensagens de outros agentes diretamente, sem precisar solicitar informações individualmente. Isso melhora

In [24]:
upcoming_steps = agent.get_upcoming_steps(task.task_id)
print(f"Num upcoming steps for task {task.task_id}: {len(upcoming_steps)}")
upcoming_steps[0]

Num upcoming steps for task 21fe1ba2-82bc-4e4e-af8f-a0aad7d53606: 1


TaskStep(task_id='21fe1ba2-82bc-4e4e-af8f-a0aad7d53606', step_id='3236a510-e5d6-475a-b39f-03453ea6cab5', input=None, step_state={}, next_steps={}, prev_steps={}, is_ready=True)

In [25]:
step_output = agent.run_step(
    task.task_id, input="What about how agents share information?"
)

Added user message to memory: What about how agents share information?
=== Calling Function ===
Calling function: vector_tool_metagpt with args: {"query": "como os agentes compartilham informa\u00e7\u00f5es no MetaGPT"}
=== Function Output ===
O MetaGPT utiliza um mecanismo de publicação-assinatura para compartilhar informações entre os agentes. Ao invés de comunicação direta entre os agentes, como em abordagens anteriores, o MetaGPT estabelece um pool de mensagens global onde os agentes podem publicar suas saídas estruturadas. Dessa forma, qualquer agente pode acessar diretamente as informações de que precisa, eliminando a necessidade de consultar outros agentes e aguardar suas respostas. Isso torna a comunicação mais eficiente.

Para evitar sobrecarga de informações, o MetaGPT também implementa um mecanismo de assinatura, onde os agentes podem selecionar as informações relevantes para suas funções com base em seus perfis de interesse. Assim, cada agente recebe apenas as informações n

In [26]:
step_output = agent.run_step(task.task_id)
print(step_output.is_last)

True


In [27]:
response = agent.finalize_response(task.task_id)

In [28]:
print(str(response))